# 1. Load Libraries

In [1]:
# Import Required Libraries
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import numpy as np
import matplotlib.pyplot as plt
import math
from math import ceil
from plotly.subplots import make_subplots
from pathlib import Path
from scipy.stats import pearsonr, theilslopes

# 2. Custom Color Palette

In [2]:
# ====== CUSTOM COLOR PALETTE ======
CUSTOM_PALETTE = [
    "#8fd7d7",  # Light Blue
    "#ff8ca1",  # Light Pink
    "#bdd373",  # Light Green
    "#ffcd8e",  # Light Orange
    "#c3aed6",  # Light Purple
    "#f9e79f",  # Light Yellow
    "#f5b7b1",  # Light Coral
    "#f45f74",  # Medium Pink
    "#98c127",  # Medium Green
    "#00b0be",  # Medium Blue
    "#ffb255",  # Medium Orange
    "#d5f4e6",  # Minty Green
    "#f7cac9",  # Very Light Pink
    "#92a8d1",  # Soft Blue
    "#aed9e0"   # Icy Blue
]

px.defaults.color_discrete_sequence = CUSTOM_PALETTE
px.defaults.template = "plotly_white"

In [3]:
palette_demo = pd.DataFrame({
    'Color_Index': range(len(CUSTOM_PALETTE)),
    'Color_Name': [f'Color {i+1}' for i in range(len(CUSTOM_PALETTE))],
    'Hex_Code': CUSTOM_PALETTE,
    'Value': [1] * len(CUSTOM_PALETTE)
})

palette_demo["Color_Index"] = palette_demo["Color_Index"].astype(str)

fig_palette = px.bar(
    palette_demo, 
    x='Color_Index', 
    y='Value',
    color='Color_Index',
    color_discrete_sequence=CUSTOM_PALETTE,
    title='Custom Color Palette - 15 Consistent Colors',
    labels={'Color_Index': 'Color Position', 'Value': 'Sample Value'},
    text='Hex_Code'
)

fig_palette.update_traces(textposition='inside', textfont_color='white')
fig_palette.update_layout(
    showlegend=False,
    height=400,
    xaxis_title='Color Position in Palette',
    yaxis_title='',
    yaxis_showticklabels=False
)

fig_palette.show()

# 3. Load Dataset

In [4]:
death_data = pd.read_csv('all_cause_of_death_data.csv')
pollutants_data = pd.read_csv('merged_pollutants_data.csv')
population_data = pd.read_csv('processed_population_data.csv')
merged = pd.read_csv('FINAL_pollutants_deaths_population.csv')

In [5]:
merged["Region Name"].unique()

array(['Southern Latin America', 'Southeast Asia', 'South Asia',
       'Caribbean', 'Western Europe', 'Western Africa', 'Oceania',
       'Central Europe', 'Southern Africa', 'Central Latin America',
       'North Africa and Middle East', 'Eastern Africa', 'Central Asia',
       'High-income Asia Pacific', 'Andean Latin America',
       'Eastern Europe', 'East Asia', 'Central Africa',
       'High-income North America', 'Tropical Latin America',
       'Australasia'], dtype=object)

Take this step to pre-processing:

In [6]:
for df in (pollutants_data, population_data):
    if 'Country' in df.columns:
        df['Country'] = df['Country'].astype(str).str.strip()
    if 'Year' in df.columns:
        df['Year'] = pd.to_numeric(df['Year'], errors='coerce')

# 4. Pollutants Analysis

As the world has industrialized and urbanized over the past three decades, a fundamental question to assess is: **Are we winning or losing the battle against air pollution?** While headlines often focus on individual cities or dramatic events, the global picture requires a data-driven lens to understand the complex relationship between pollution exposure, population dynamics and human health outcomes.

This analysis employs **population-weighted metrics** rather than simple country averages — a crucial methodological choice that ensures our insights reflect human impact, not just geography. When we say "global air quality is improving," we mean the average person's exposure is declining, not just that more countries happen to have better air.

By the end of this analysis, we'll know not just whether air quality is improving globally, but **where** progress is happening, **what** drives it, and how it translates into human health outcomes.

## 4.1 Global Population-Weighted Pollutants Exposure Line Chart


### 1. Global Population-Weighted Exposure

For each year \(y\) and pollutant \(p\):

$$
\text{Exposure\_PW}_{y,p}
= \frac{\sum\limits_{c} \text{Exposure}_{c,y,p} \cdot \text{Population}_{c,y}}
       {\sum\limits_{c} \text{Population}_{c,y}}
$$

**Why**: ensures global averages reflect population size — large populations influence the global mean more than small ones.

---

### 2. Indexed Exposure (1990 = 100)


Index for year \(y\):

$$
\text{Index}_{y,p}
= \frac{\text{Exposure\_PW}_{y,p}}{\text{Exposure\_PW}_{1990,p}} \times 100
$$

**Why**: normalizes exposures across pollutants with different units/scales, enabling direct trend comparisons (e.g., “% change relative to 1990”).


In [7]:
merged = pollutants_data.merge(
    population_data[['Country', 'Year', 'Population']],
    on=['Country', 'Year'],
    how='left',
    validate='m:1'
)
merged = merged.dropna(subset=['Population'])

col_map = {
    'Exposure Mean PM25':  'PM2.5',
    'Exposure Mean NO2':   'NO2',
    'Exposure Mean OZONE': 'O3',
    'Exposure Mean HAP':   'HAP',
}

present = {k: v for k, v in col_map.items() if k in merged.columns}

if not present:
    raise KeyError("None of the expected exposure columns found in pollutants_data.")

exposures = (
    merged[['Year', 'Population', *present.keys()]]
      .rename(columns=present)
      .melt(id_vars=['Year', 'Population'],
            var_name='pollutant',
            value_name='Exposure')
      .dropna(subset=['Exposure'])
)

# --- Global population-weighted exposure per year & pollutant ---
agg = (
    exposures
    .assign(wx=lambda d: d['Exposure'] * d['Population'])
    .groupby(['Year', 'pollutant'], as_index=False)
    .agg(wx=('wx', 'sum'), w=('Population', 'sum'))
)
global_pw = agg.assign(Exposure_PW=agg['wx'] / agg['w'])[['Year', 'pollutant', 'Exposure_PW']]

base = (
    global_pw[global_pw['Year'] == 1990][['pollutant', 'Exposure_PW']]
    .rename(columns={'Exposure_PW': 'base'})
)

global_pw = global_pw.merge(base, on='pollutant', how='left')
global_pw['Index'] = (global_pw['Exposure_PW'] / global_pw['base']) * 100.0
global_pw = global_pw.drop(columns='base')




In [8]:
a1 = (global_pw.loc[global_pw['Year'] == 2020, ['pollutant', 'Index']]
                .set_index('pollutant')
                .rename(columns={'Index': 'Change vs 1990 (%)'}))

a1['Change vs 1990 (%)'] = a1['Change vs 1990 (%)'] - 100
a1 = a1.sort_values('Change vs 1990 (%)')
a1

,Change vs 1990 (%)
pollutant,
HAP,-36.349208
PM2.5,-21.028208
NO2,-17.753072
O3,13.515145


In [9]:
fig = px.line(
    global_pw, x='Year', y='Index', color='pollutant',
    title='Global Population-Weighted Exposure (Index, 1990 = 100)',
    labels={'Index': 'Index (1990 = 100)', 'pollutant': 'Pollutant'}
)
fig.add_hline(
    y=100, line_dash='dash', line_color='gray',
    annotation_text='100 = 1990 baseline', annotation_position='top left'
)
fig.update_layout(legend_title_text='Pollutant')
fig.show()

### Key Findings

After three decades of environmental policies, economic development and technological innovation, the answer is **not straightforward** — because different pollutants tell different stories. Success with one pollutant doesn't guarantee success with others.

| Pollutant | Change (1990–2020) | Interpretation |
|-----------|-------------------|----------------|
| **HAP** | **-36.34%** | Public health's greatest victory — hundreds of millions gained access to clean cooking fuels, driven by rural electrification and LPG subsidies such as India's Ujjwala Yojana (PMUY). |
| **PM2.5** | **-21.02%** | Reflects successful implementation of air quality standards, industrial controls, and better urban planning. |
| **NO₂** | **-17.75%** | Driven by stricter Euro standards, Clean Air Act regulations, and the rise of electric vehicles. |
| **O₃** | **+13.51%** | The outlier — ozone worsens, revealing the chemical complexity of air quality management. |

The global PM2.5 shows a slight increase in early 2000s before a sharp drop after 2013. The initial bump corresponds to surging industrial growth in rapidly developing nations (particularly in Asia), followed by improvements as pollution control measures took effect. China's aggressive "Air Pollution Prevention and Control Action Plan" of 2013 likely contributed to the post-2013 global PM2.5 decline — China's average PM2.5 exposure fell from 49.3 to 34.8 µg/m³ (a 29.4% decrease). However, current levels in many countries still vastly exceed the WHO guideline of 5 µg/m³.

**Ozone** reveals a different pattern: unlike other pollutants which showed significant improvements, ozone levels have remained upward. Ozone is a secondary pollutant formed through complex photochemical reactions involving precursor emissions (NOx and VOCs) and influenced by temperature and sunlight intensity. Countries in Asia like Bangladesh, India, Indonesia, Pakistan & China notably contributed, potentially driven by rising temperatures due to climate change and increased urbanization.

**NO₂** demonstrates a complex pattern with a relatively stable period in the 1990s, a notable peak around 2005–2010, and a significant decline after 2010. The peak corresponds to rapid industrialization in developing countries, while the post-2010 decline reflects stricter vehicle emission standards (Euro 6, Tier 3), adoption of catalytic converters, shifts toward renewable energy, and targeted urban air quality management.

**HAP's** steady global decline indicates substantial progress in reducing indoor air pollution worldwide, with Brazil having the largest percentage reduction. This is particularly significant as HAP primarily affects rural and low-income populations.

## 4.2 Parity Plots (Pollutants 1990 VS 2020)

Since each region's pollution sources vary significantly, it is essential to analyse pollutants at a regional level. For example, in South Asia air pollution is heavily linked to older vehicles and weak emissions standards; in the Amazon basin, mass forest fires are the dominant source; and in Sub-Saharan Africa, heavy industrial activity and reliance on low-quality fuels are primary contributors.

The parity plots below compare each country's pollutant exposure in **1990 (x-axis)** versus **2020 (y-axis)**, grouped by canonical region. The bubble colour is region-coded, the size reflects population, and the **diagonal parity line** marks no-change — every bubble **below** the line represents improvement, and every bubble **above** represents deterioration.

In [10]:
"""Map detailed region names to broader regional categories for faceting."""
REGION_MAP = {
    # Africa
    'Central Africa': 'Africa',
    'Eastern Africa': 'Africa',
    'Southern Africa': 'Africa',
    'Western Africa': 'Africa',
    'North Africa and Middle East': 'MENA',
    # Americas
    'Caribbean': 'Americas',
    'Andean Latin America': 'Americas',
    'Central Latin America': 'Americas',
    'Southern Latin America': 'Americas',
    'Tropical Latin America': 'Americas',
    'High-income North America': 'Americas',
    # Asia
    'Central Asia': 'Asia',
    'East Asia': 'Asia',
    'South Asia': 'Asia',
    'Southeast Asia': 'Asia',
    'High-income Asia Pacific': 'Asia',
    # Europe
    'Central Europe': 'Europe',
    'Eastern Europe': 'Europe',
    'Western Europe': 'Europe',
    # Oceania
    'Oceania': 'Oceania',
    'Australasia': 'Oceania',
}

def canonical_region(region_name: str) -> str:
    return REGION_MAP.get(region_name, 'Other')


In [11]:
_base = merged.loc[
    merged['Year'].isin([1990, 2020]),
    ['Country', 'Region Name', 'Year', 'Population', 'Exposure Mean PM25']
]
wide = _base.pivot_table(
    index=['Country', 'Region Name'],
    columns='Year',
    values=['Exposure Mean PM25', 'Population'],
    aggfunc='first'
)
wide.columns = [('PM25_'+str(c[1]) if c[0] == 'Exposure Mean PM25' else 'Pop_'+str(c[1])) for c in wide.columns]
wide = wide.reset_index().dropna(subset=['PM25_1990', 'PM25_2020'])
wide['Canonical_Region'] = wide['Region Name'].apply(canonical_region)
wide['size'] = np.sqrt(wide['Pop_2020'].clip(lower=0)) if 'Pop_2020' in wide.columns else 10

lo = float(min(wide[['PM25_1990', 'PM25_2020']].min()))
hi = float(max(wide[['PM25_1990', 'PM25_2020']].max()))
parity_line = [lo, hi]

fig = px.scatter(
    wide, x='PM25_1990', y='PM25_2020', color='Canonical_Region', size='size',
    hover_name='Country',
    hover_data={'Region Name': True, 'Canonical_Region': True},
    labels={'PM25_1990': 'PM2.5 in 1990 (μg/m³)', 'PM25_2020': 'PM2.5 in 2020 (μg/m³)', 'Canonical_Region': 'Region'},
    title='PM2.5 Parity Plot by Country: 1990 vs 2020 (by Canonical Region)'
)
fig.add_trace(go.Scatter(x=parity_line, y=parity_line, mode='lines', name='Parity (y=x)', line=dict(dash='dash', color='black')))
fig.update_traces(marker=dict(opacity=0.8, line=dict(width=0)))
fig.show()

In [12]:
_base = merged.loc[
    merged['Year'].isin([1990, 2020]),
    ['Country', 'Region Name', 'Year', 'Population', 'Exposure Mean HAP']
]
wide = _base.pivot_table(
    index=['Country', 'Region Name'],
    columns='Year',
    values=['Exposure Mean HAP', 'Population'],
    aggfunc='first'
)
wide.columns = [('HAP_'+str(c[1]) if c[0] == 'Exposure Mean HAP' else 'Pop_'+str(c[1])) for c in wide.columns]
wide = wide.reset_index().dropna(subset=['HAP_1990', 'HAP_2020'])
wide['Canonical_Region'] = wide['Region Name'].apply(canonical_region)
wide['size'] = np.sqrt(wide['Pop_2020'].clip(lower=0)) if 'Pop_2020' in wide.columns else 10

lo = float(min(wide[['HAP_1990', 'HAP_2020']].min()))
hi = float(max(wide[['HAP_1990', 'HAP_2020']].max()))
parity_line = [lo, hi]

fig = px.scatter(
    wide, x='HAP_1990', y='HAP_2020', color='Canonical_Region', size='size',
    hover_name='Country',
    hover_data={'Region Name': True, 'Canonical_Region': True},
    labels={'HAP_1990': 'HAP in 1990 (μg/m³)', 'HAP_2020': 'HAP in 2020 (μg/m³)', 'Canonical_Region': 'Region'},
    title='HAP Parity Plot by Country: 1990 vs 2020 (by Canonical Region)'
)
fig.add_trace(go.Scatter(x=parity_line, y=parity_line, mode='lines', name='Parity (y=x)', line=dict(dash='dash', color='black')))
fig.update_traces(marker=dict(opacity=0.8, line=dict(width=0)))
fig.show()


In [13]:
_base = merged.loc[
    merged['Year'].isin([1990, 2020]),
    ['Country', 'Region Name', 'Year', 'Population', 'Exposure Mean NO2']
]
wide = _base.pivot_table(
    index=['Country', 'Region Name'],
    columns='Year',
    values=['Exposure Mean NO2', 'Population'],
    aggfunc='first'
)
wide.columns = [('NO2_'+str(c[1]) if c[0] == 'Exposure Mean NO2' else 'Pop_'+str(c[1])) for c in wide.columns]
wide = wide.reset_index().dropna(subset=['NO2_1990', 'NO2_2020'])
wide['Canonical_Region'] = wide['Region Name'].apply(canonical_region)
wide['size'] = np.sqrt(wide['Pop_2020'].clip(lower=0)) if 'Pop_2020' in wide.columns else 10

lo = float(min(wide[['NO2_1990', 'NO2_2020']].min()))
hi = float(max(wide[['NO2_1990', 'NO2_2020']].max()))
parity_line = [lo, hi]

fig = px.scatter(
    wide, x='NO2_1990', y='NO2_2020', color='Canonical_Region', size='size',
    hover_name='Country',
    hover_data={'Region Name': True, 'Canonical_Region': True},
    labels={'NO2_1990': 'NO₂ in 1990 (μg/m³)', 'NO2_2020': 'NO₂ in 2020 (μg/m³)', 'Canonical_Region': 'Region'},
    title='NO₂ Parity Plot by Country: 1990 vs 2020 (by Canonical Region)'
)
fig.add_trace(go.Scatter(x=parity_line, y=parity_line, mode='lines', name='Parity (y=x)', line=dict(dash='dash', color='black')))
fig.update_traces(marker=dict(opacity=0.8, line=dict(width=0)))
fig.show()


In [14]:
_base = merged.loc[
    merged['Year'].isin([1990, 2020]),
    ['Country', 'Region Name', 'Year', 'Population', 'Exposure Mean OZONE']
]
wide = _base.pivot_table(
    index=['Country', 'Region Name'],
    columns='Year',
    values=['Exposure Mean OZONE', 'Population'],
    aggfunc='first'
)
wide.columns = [('O3_'+str(c[1]) if c[0] == 'Exposure Mean OZONE' else 'Pop_'+str(c[1])) for c in wide.columns]
wide = wide.reset_index().dropna(subset=['O3_1990', 'O3_2020'])
wide['Canonical_Region'] = wide['Region Name'].apply(canonical_region)
wide['size'] = np.sqrt(wide['Pop_2020'].clip(lower=0)) if 'Pop_2020' in wide.columns else 10

lo = float(min(wide[['O3_1990', 'O3_2020']].min()))
hi = float(max(wide[['O3_1990', 'O3_2020']].max()))
parity_line = [lo, hi]

fig = px.scatter(
    wide, x='O3_1990', y='O3_2020', color='Canonical_Region', size='size',
    hover_name='Country',
    hover_data={'Region Name': True, 'Canonical_Region': True},
    labels={'O3_1990': 'O₃ in 1990 (ppb)', 'O3_2020': 'O₃ in 2020 (ppb)', 'Canonical_Region': 'Region'},
    title='O₃ (Ozone) Parity Plot by Country: 1990 vs 2020 (by Canonical Region)'
)
fig.add_trace(go.Scatter(x=parity_line, y=parity_line, mode='lines', name='Parity (y=x)', line=dict(dash='dash', color='black')))
fig.update_traces(marker=dict(opacity=0.8, line=dict(width=0)))
fig.show()


### Regional Insights from Parity Plots

**Africa:** In almost every pollutant, African regions are clustered above parity with worrying bubbles. The increase of PM2.5 & NO₂ is worsening due to rapid urbanization, industrial growth, weak emission standards, and reliance on low-quality fuels. O₃ increases can be linked to more vehicles and unchecked NOx/VOCs. Africa is at a tipping point — without proactive regulations, worsening air pollution will severely affect rapidly urbanizing populations. Investments in clean transport, fuels, and industrial emissions standards are urgent.

**Asia:** Real reduction despite growth, driven by China's clean air reforms and India's gradual vehicle standards. Asia is proof that improvement is possible even with massive population and economic growth. Replicating China's regulatory playbook could accelerate gains elsewhere. However, pollution levels remain far beyond WHO guidelines.

**Europe & Americas:** Consistently stayed below parity, with levels already close to WHO limits in 1990 — demonstrating that long-term regulatory consistency (air quality policies, vehicle standards, industrial regulation) worked. These can serve as models for other regions.

**MENA:** A divided picture — some wealthy Gulf states are improving, while others worsen due to oil combustion, motorization and dust storms. Ozone is variable but trending higher in the majority of areas.

**Oceania:** PM2.5, NO₂ and HAPs were mostly low to begin with, with modest further reductions. O₃ fluctuations are tied more to natural conditions and imported pollution than local sources. Oceania generally performs well, benefiting from lower industrialization and stricter environmental regimes.

## 4.3 Quadrant Plots

To see how on a global level countries are performing, each scatter plot below showcases each country's **population change** and **pollutant change** from 1990 to 2020. Countries are grouped into four quadrants:

1. **Population ↑ & Pollutant ↑ (Red)** — Compounding risk: more people exposed to worsening air.
2. **Population ↑ & Pollutant ↓ (Green)** — The success case: growth with cleaner air.
3. **Population ↓ & Pollutant ↓ (Blue)** — Broad improvement, often from structural shifts or deindustrialization.
4. **Population ↓ & Pollutant ↑ (Orange)** — Concerning anomaly: shrinking population but deteriorating air quality.

In [15]:
pop_df = (
population_data
    .loc[:, ["Country", "Year", "Population"]]
    .assign(Country=lambda d: d["Country"].astype(str))
)

pol_df = (
pollutants_data
    .rename(
        columns={
            "Exposure Mean PM25": "PM25",
            "Exposure Mean NO2": "NO2",
            "Exposure Mean OZONE": "O3",
            "Exposure Mean HAP": "HAP",
            "Region Name": "Region",
        }
    )
    .loc[:, ["Country", "Region", "Year", "PM25", "NO2", "O3", "HAP"]]
    .assign(Country=lambda d: d["Country"].astype(str))
)

# === 1) Build local, narrowed views without mutating the originals ===
pop_df = (
    population_data
        .rename(columns={"Value": "Population", "year": "Year", "country": "Country"})
        .loc[:, ["Country", "Year", "Population"]]
        .assign(Country=lambda d: d["Country"].astype(str))
)

pol_df = (
    pollutants_data
        .rename(
            columns={
                "Exposure Mean PM25": "PM25",
                "Exposure Mean NO2": "NO2",
                "Exposure Mean OZONE": "O3",
                "Exposure Mean HAP": "HAP",
                "Region Name": "Region",
            }
        )
        .loc[:, ["Country", "Region", "Year", "PM25", "NO2", "O3", "HAP"]]
        .assign(Country=lambda d: d["Country"].astype(str))
)


years = [1990, 2020]
pop_yrs = pop_df[pop_df["Year"].isin(years)]
pol_yrs = pol_df[pol_df["Year"].isin(years)]


# Countries present in BOTH years in BOTH datasets
pop_valid = (
    pop_yrs.groupby("Country", observed=True)["Year"].nunique().loc[lambda s: s == 2].index
)
pol_valid = (
    pol_yrs.groupby("Country", observed=True)["Year"].nunique().loc[lambda s: s == 2].index
)
valid_countries = set(pop_valid).intersection(pol_valid)

pop_yrs = pop_yrs[pop_yrs["Country"].isin(valid_countries)]
pol_yrs = pol_yrs[pol_yrs["Country"].isin(valid_countries)]

In [16]:
# === 3) Compute changes ===
# Population % change: (2020-1990)/1990 * 100
pop_pivot = pop_yrs.pivot_table(index="Country", columns="Year", values="Population")
pop_change = ((pop_pivot[2020] - pop_pivot[1990]) / pop_pivot[1990]) * 100
pop_change.name = "Population % Change"

# Pollutant absolute deltas: 2020 - 1990
pollutant_list = ["PM25", "NO2", "O3", "HAP"]
pol_change_df = pd.DataFrame(index=pop_change.index)
for p in pollutant_list:
    p_pivot = pol_yrs.pivot_table(index="Country", columns="Year", values=p)
    pol_change_df[f"Δ{p}"] = p_pivot[2020] - p_pivot[1990]

# Attach one Region per country (prefer 2020)
region_map = (
    pol_yrs.sort_values(["Country", "Year"])
           .drop_duplicates("Country", keep="last")
           .set_index("Country")["Region"]
)

act4_data = (
    pd.concat([pop_change, pol_change_df], axis=1)
      .join(region_map, how="left")
      .reset_index()
)


PALETTE = {
    "quad_q1": "#ffcccc",  # +x, +y
    "quad_q4": "#ccffcc",  # +x, -y
    "quad_q3": "#cce5ff",  # -x, -y
    "quad_q2": "#ffe5cc",  # -x, +y
    "marker": "rgba(0,0,0,0.7)",
    "grid": "gray",
}

def make_quadrant_figure(df, y_col, title_text, y_axis_title, hover_unit_fmt):
    sub = df.dropna(subset=["Population % Change", y_col]).copy()

    up = sub["Population % Change"] >= 0
    right = sub[y_col] >= 0
    sub["Quadrant"] = np.select(
        [up & right, up & ~right, ~up & ~right, ~up & right],
        [f"Pop↑ & {y_col[1:]}↑", f"Pop↑ & {y_col[1:]}↓", f"Pop↓ & {y_col[1:]}↓", f"Pop↓ & {y_col[1:]}↑"],
        default="Unknown",
    )

    quad_counts = sub["Quadrant"].value_counts().to_dict()
    quad_text = "<br>".join(f"• {k}: {v}" for k, v in sorted(quad_counts.items()))

    x_min, x_max = sub["Population % Change"].min(), sub["Population % Change"].max()
    y_min, y_max = sub[y_col].min(), sub[y_col].max()

    fig = go.Figure()
    fig.add_shape(type="rect", x0=0, x1=x_max, y0=0, y1=y_max,
                  fillcolor=PALETTE["quad_q1"], opacity=0.25, line_width=0)
    fig.add_shape(type="rect", x0=0, x1=x_max, y0=y_min, y1=0,
                  fillcolor=PALETTE["quad_q4"], opacity=0.25, line_width=0)
    fig.add_shape(type="rect", x0=x_min, x1=0, y0=y_min, y1=0,
                  fillcolor=PALETTE["quad_q3"], opacity=0.25, line_width=0)
    fig.add_shape(type="rect", x0=x_min, x1=0, y0=0, y1=y_max,
                  fillcolor=PALETTE["quad_q2"], opacity=0.25, line_width=0)

    fig.add_trace(
        go.Scatter(
            x=sub["Population % Change"], y=sub[y_col],
            mode="markers", marker=dict(color=PALETTE["marker"], size=8),
            text=sub["Country"],
            hovertemplate="%{text}<br>Pop %Δ: %{x:.1f}%<br>"
                          f"{y_col[1:]} Δ: %{{y:{hover_unit_fmt}}}<extra></extra>",
        )
    )
    fig.add_hline(y=0, line_color=PALETTE["grid"], line_dash="dash")
    fig.add_vline(x=0, line_color=PALETTE["grid"], line_dash="dash")

    fig.update_layout(
        height=600,
        title=dict(text=title_text, y=0.95, x=0.5, xanchor="center", yanchor="top", font=dict(size=14)),
        annotations=[
            dict(x=1.0, y=0.98, xref="paper", yref="paper",
                 text=f"<b>Quadrant Counts:</b><br>{quad_text}", showarrow=False,
                 font=dict(size=10), align="left",
                 bgcolor="rgba(255,255,255,0.8)", bordercolor="gray", borderwidth=1, borderpad=4)
        ],
        margin=dict(t=100, l=80, r=40, b=80),
    )
    fig.update_xaxes(title_text="Population % Change (1990 → 2020)", title_font=dict(size=12), tickfont=dict(size=10))
    fig.update_yaxes(title_text=y_axis_title, title_font=dict(size=12), tickfont=dict(size=10))
    return fig

In [17]:
fig_d1 = make_quadrant_figure(
    act4_data,
    y_col="ΔPM25",
    title_text="Figure D1 — Quadrant Map (PM2.5): Population vs Exposure Change (1990–2020)",
    y_axis_title="PM2.5 Change (2020 − 1990, µg/m³)",
    hover_unit_fmt=".1f",
)
fig_d1

In [18]:
fig_d2 = make_quadrant_figure(
    act4_data,
    y_col="ΔNO2",
    title_text="Figure D2 — Quadrant Map (NO₂): Population vs Exposure Change (1990–2020)",
    y_axis_title="NO₂ Change (2020 − 1990, µg/m³)",
    hover_unit_fmt=".1f",
)
fig_d2

In [19]:
fig_d3 = make_quadrant_figure(
    act4_data,
    y_col="ΔO3",
    title_text="Figure D3 — Quadrant Map (O₃): Population vs Exposure Change (1990–2020)",
    y_axis_title="O₃ Change (2020 − 1990, ppb)",
    hover_unit_fmt=".1f",
)
fig_d3

In [20]:
fig_d4 = make_quadrant_figure(
    act4_data,
    y_col="ΔHAP",
    title_text="Figure D4 — Quadrant Map (HAP): Population vs Exposure Change (1990–2020)",
    y_axis_title="HAP Change (2020 − 1990, proportion)",
    hover_unit_fmt=".2f",
)
fig_d4

### Quadrant Analysis Results

Across all pollutants, the **green quadrant** (success case) has the largest share:
- **PM2.5:** 118 countries showing improvement alongside growth
- **O₃:** 63 countries (fewest success stories, reflecting its complex atmospheric chemistry)
- **NO₂:** 102 countries
- **HAP:** 158 countries — the largest success story, reflecting clean cooking initiatives and energy transitions

HAP shows the most widespread improvements even in growing populations. PM2.5 and NO₂ show mixed success, with some growing countries improving and others worsening. O₃ presents the most challenging pattern.

There's also a small but worrying **orange quadrant** (shrinking population, worsening air): 4 countries for PM2.5, 5 for O₃, 3 for NO₂, and only 1 for HAP — suggesting possible legacy pollution sources, poor regulations, or specific industrial sectors dominating the economy.

## 4.4 Combined Pollution Index (CPI) based World 

It aggregates multiple pollutants into a single composite score, while keeping each pollutant dimensionless and comparable via normalization, Countries with high exposures across multiple pollutants will score higher, highlighting overall pollution burden rather than focusing on one pollutant.


### **Combined Pollution Index (CPI) Formula**

### Step 1: Normalize each pollutant  
For pollutant $(p)  \text{ in \{\text{PM2.5}, \text{NO₂}, \text{O₃}, \text{HAP}\}}$, country $(c)$, and year $(y)$:

$$
X^{\text{norm}}_{c,y,p} \;=\; 
\frac{X_{c,y,p} - \min\limits_{c',y'} X_{c',y',p}}
     {\max\limits_{c',y'} X_{c',y',p} - \min\limits_{c',y'} X_{c',y',p}}
$$

where $(X_{c,y,p})$ is the raw exposure, and the min/max are taken globally across all countries and years.

---

### Step 2: Compute CPI as the sum of normalized pollutants  
$$
\text{CPI}_{c,y} \;=\; \sum_{p \in P} X^{\text{norm}}_{c,y,p}
$$

where $(P)$ is the set of pollutants available.  
- If all four pollutants are present, $(\text{CPI}_{c,y} \in [0,4])$.  
- If fewer pollutants are present, the maximum possible CPI equals $(|P|)$.

In [21]:
pollutant_cols = ['Exposure Mean HAP', 'Exposure Mean NO2', 'Exposure Mean OZONE', 'Exposure Mean PM25']
available_cols = [c for c in pollutant_cols if c in merged.columns]

# Min-max normalize each pollutant globally (across all countries/years)
for col in available_cols:
    cmin = merged[col].min()
    cmax = merged[col].max()
    merged[f'{col}_normalized'] = (merged[col] - cmin) / (cmax - cmin)

norm_cols = [f'{c}_normalized' for c in available_cols]

# Combined Pollution Index: sum of normalized pollutants
merged['Combined_Pollution_Index'] = merged[norm_cols].sum(axis=1)

desired_years = [1990, 2000, 2010, 2020]
df_maps = merged.loc[merged['Year'].isin(desired_years), ['Country', 'Year', 'Combined_Pollution_Index', *available_cols]]

vmin = float(df_maps['Combined_Pollution_Index'].min())
vmax = float(df_maps['Combined_Pollution_Index'].max())

In [22]:
fig = px.choropleth(
    df_maps,
    locations='Country',
    locationmode='country names',
    color='Combined_Pollution_Index',
    hover_name='Country',
    hover_data={
        'Combined_Pollution_Index': ':.2f',
        'Exposure Mean PM25': ':.1f' if 'Exposure Mean PM25' in available_cols else False,
        'Exposure Mean NO2': ':.1f' if 'Exposure Mean NO2' in available_cols else False,
        'Exposure Mean OZONE': ':.1f' if 'Exposure Mean OZONE' in available_cols else False,
        'Exposure Mean HAP': ':.1f' if 'Exposure Mean HAP' in available_cols else False,
    },
    facet_col='Year',
    facet_col_wrap=2,
    facet_col_spacing=0.03,
    facet_row_spacing=0.03,
    color_continuous_scale=[
        [0.0, CUSTOM_PALETTE[8]],
        [0.25, CUSTOM_PALETTE[5]],
        [0.5, CUSTOM_PALETTE[3]],
        [0.75, CUSTOM_PALETTE[1]],
        [1.0, CUSTOM_PALETTE[7]],
    ],
    range_color=[vmin, vmax],
    category_orders={'Year': desired_years},
    title='Combined Pollution Index — Global Snapshots (1990, 2000, 2010, 2020)<br><sub>Normalized sum of PM2.5, NO2, Ozone, and HAP exposure</sub>'
)

fig.for_each_annotation(lambda a: a.update(text=a.text.split('=')[-1]))
fig.update_coloraxes(colorbar_title='Combined Pollution<br>Index (0–4 scale)')


rows = int(ceil(len(desired_years) / 2))
fig.update_layout(width=1000, height=420 * rows, title_x=0.5, font=dict(size=12))

fig.show()

/var/folders/82/w51l4pcn5351zd0bz7gqwcrr0000gq/T/ipykernel_37553/2222711615.py:1: DeprecationWarning: The library used by the *country names* `locationmode` option is changing in an upcoming version. Country names in existing plots may not work in the new version. To ensure consistent behavior, consider setting `locationmode` to *ISO-3*.
  fig = px.choropleth(


In [23]:
latest_year = max(desired_years)
latest_data = df_maps[df_maps['Year'] == latest_year].sort_values('Combined_Pollution_Index', ascending=False)
top_10 = latest_data.head(10)

print(f"\nTop 10 countries by Combined Pollution Index in {latest_year}:")
for rank, (_, row) in enumerate(top_10.iterrows(), start=1):
    extras = []
    if 'Exposure Mean PM25' in available_cols: extras.append(f"PM2.5: {row['Exposure Mean PM25']:5.1f}")
    if 'Exposure Mean NO2' in available_cols: extras.append(f"NO2: {row['Exposure Mean NO2']:5.1f}")
    if 'Exposure Mean OZONE' in available_cols: extras.append(f"O3: {row['Exposure Mean OZONE']:5.1f}")
    if 'Exposure Mean HAP' in available_cols: extras.append(f"HAP: {row['Exposure Mean HAP']:5.1f}")
    extras_str = ", ".join(extras)
    print(f"{rank:2d}. {row['Country']:<22s} CPI: {row['Combined_Pollution_Index']:5.2f}  ({extras_str})")


Top 10 countries by Combined Pollution Index in 2020:
 1. Niger                  CPI:  2.56  (PM2.5:  85.1, NO2:   5.6, O3:  48.6, HAP:   1.0)
 2. Qatar                  CPI:  2.37  (PM2.5:  75.7, NO2:  28.9, O3:  67.6, HAP:   0.0)
 3. Bangladesh             CPI:  2.32  (PM2.5:  42.4, NO2:  11.5, O3:  65.4, HAP:   0.7)
 4. India                  CPI:  2.30  (PM2.5:  48.4, NO2:  12.1, O3:  67.2, HAP:   0.6)
 5. Nepal                  CPI:  2.28  (PM2.5:  45.7, NO2:   9.2, O3:  67.5, HAP:   0.7)
 6. Mali                   CPI:  2.25  (PM2.5:  56.8, NO2:   6.8, O3:  45.4, HAP:   1.0)
 7. Benin                  CPI:  2.24  (PM2.5:  51.0, NO2:   4.7, O3:  53.5, HAP:   0.9)
 8. Burkina Faso           CPI:  2.24  (PM2.5:  58.5, NO2:   5.5, O3:  47.9, HAP:   0.9)
 9. Togo                   CPI:  2.23  (PM2.5:  51.7, NO2:   5.4, O3:  53.2, HAP:   0.9)
10. Gambia                 CPI:  2.23  (PM2.5:  58.4, NO2:   7.4, O3:  44.3, HAP:   0.9)


## 4.5 Population-Weighted Pollution Analysis (Total Human Impact)

### Total Human Impact (Population-Weighted Pollution Burden)

**Definition:**  
*Total Human Impact* measures the overall pollution burden on a country’s population.  
It combines **pollution intensity** (CPI) with **population size**, capturing how many people are exposed and to what degree.

---

### 1. Raw Formula

For a given country $c$ in year $y$:

$$
\text{Total\_Human\_Impact}_{c,y} = \text{CPI}_{c,y} \times \text{Population}_{c,y}
$$

where:  
- $\text{CPI}_{c,y}$ = Combined Pollution Index for country $c$ in year $y$  
- $\text{Population}_{c,y}$ = total population of country $c$ in year $y$  

---

### 2. Log-Scaled and Min-Max Normalized Version

To handle skewness and outliers (e.g., China, India), we apply a log transformation followed by min-max scaling to the range $[0,100]$.

1. **Log transformation** (using $\log(1+x)$ to handle zero values):

$$
L_{c,y} = \log \big( 1 + \text{Total\_Human\_Impact}_{c,y} \big)
$$

2. **Min-max scaling** across all countries and years:

$$
\text{Scaled\_Impact}_{c,y} \;=\; 
\frac{L_{c,y} - \min\limits_{c',y'} L_{c',y'}}{\max\limits_{c',y'} L_{c',y'} - \min\limits_{c',y'} L_{c',y'}} \times 100
$$

---

### Interpretation
- **$100$** → Country with the maximum global human impact (log-transformed).  
- **$0$** → Country with the minimum global human impact.  
- **$50$** → Median impact after log-scaling.  
- This transformation compresses extreme outliers while spreading mid-range values, making global comparisons more interpretable.


In [24]:
latest_year = 2020

latest = (
    merged.loc[merged['Year'] == latest_year, ['Country', 'Population', 'Combined_Pollution_Index',
                                               'Exposure Mean PM25', 'Exposure Mean NO2',
                                               'Exposure Mean OZONE', 'Exposure Mean HAP']]
          .dropna(subset=['Population'])
)

# Total burden = CPI × Population
latest['Total_Pollution_Burden'] = latest['Combined_Pollution_Index'] * latest['Population']

# Log1p to handle skew/outliers
latest['Log_Pollution_Burden'] = np.log1p(latest['Total_Pollution_Burden'])

# min max scaling
log_min = latest['Log_Pollution_Burden'].min()
log_max = latest['Log_Pollution_Burden'].max()
scale_den = (log_max - log_min) if (log_max > log_min) else 1.0
latest['Scaled_Pollution_Burden'] = ((latest['Log_Pollution_Burden'] - log_min) / scale_den) * 100.0

latest = latest.sort_values('Scaled_Pollution_Burden', ascending=False)
latest['Total_Burden_Rank'] = np.arange(1, len(latest) + 1)

latest['CPI_Rank'] = latest['Combined_Pollution_Index'].rank(method='min', ascending=False).astype(int)


print(f"\n🌍 TOP 10 HIGHEST TOTAL HUMAN IMPACT (Scaled 0–100) — {latest_year}")
top10 = latest.head(10)
for _, r in top10.iterrows():
    print(
        f"{int(r['Total_Burden_Rank']):2d}. {r['Country']:<25} "
        f"Scaled Burden: {r['Scaled_Pollution_Burden']:5.1f}  "
        f"(Raw: {r['Total_Pollution_Burden']:>12,.0f}, "
        f"Pop: {r['Population']/1e6:>6.1f}M, "
        f"CPI: {r['Combined_Pollution_Index']:>4.2f}, "
        f"CPI Rank: {r['CPI_Rank']:>3d})"
    )


🌍 TOP 10 HIGHEST TOTAL HUMAN IMPACT (Scaled 0–100) — 2020
 1. India                     Scaled Burden: 100.0  (Raw: 3,223,686,865, Pop: 1402.6M, CPI: 2.30, CPI Rank:   4)
 2. China                     Scaled Burden:  97.5  (Raw: 2,216,002,725, Pop: 1411.1M, CPI: 1.57, CPI Rank:  52)
 3. Pakistan                  Scaled Burden:  87.1  (Raw:  483,704,454, Pop:  235.0M, CPI: 2.06, CPI Rank:  20)
 4. Nigeria                   Scaled Burden:  86.8  (Raw:  459,972,055, Pop:  214.0M, CPI: 2.15, CPI Rank:  11)
 5. Bangladesh                Scaled Burden:  85.6  (Raw:  386,468,549, Pop:  166.3M, CPI: 2.32, CPI Rank:   3)
 6. United States of America  Scaled Burden:  84.1  (Raw:  309,691,373, Pop:  331.5M, CPI: 0.93, CPI Rank: 137)
 7. Indonesia                 Scaled Burden:  83.2  (Raw:  271,754,947, Pop:  274.8M, CPI: 0.99, CPI Rank: 126)
 8. Ethiopia                  Scaled Burden:  81.8  (Raw:  223,374,675, Pop:  118.9M, CPI: 1.88, CPI Rank:  31)
 9. Brazil                    Scaled Burden

In [25]:
desired_years = [1990, 2000, 2010, 2020]
map_df = merged.loc[merged['Year'].isin(desired_years), ['Country','Year','Population','Combined_Pollution_Index',
                                                         'Exposure Mean PM25','Exposure Mean NO2','Exposure Mean OZONE','Exposure Mean HAP']]

map_df = map_df.dropna(subset=['Population']).copy()
map_df['Total_Pollution_Burden'] = map_df['Combined_Pollution_Index'] * map_df['Population']
map_df['Log_Pollution_Burden'] = np.log1p(map_df['Total_Pollution_Burden'])

log_min_g = map_df['Log_Pollution_Burden'].min()
log_max_g = map_df['Log_Pollution_Burden'].max()
scale_den_g = (log_max_g - log_min_g) if (log_max_g > log_min_g) else 1.0
map_df['Scaled_Pollution_Burden'] = ((map_df['Log_Pollution_Burden'] - log_min_g) / scale_den_g) * 100.0

map_df['Year'] = pd.Categorical(map_df['Year'], categories=desired_years, ordered=True)

fig = px.choropleth(
    map_df,
    locations='Country',
    locationmode='country names',
    color='Scaled_Pollution_Burden',
    hover_name='Country',
    hover_data={
        'Scaled_Pollution_Burden': ':.1f',
        'Total_Pollution_Burden': ':,.0f',
        'Combined_Pollution_Index': ':.2f',
        'Population': ':,.0f',
        'Exposure Mean PM25': ':.1f' if 'Exposure Mean PM25' in map_df.columns else False,
        'Exposure Mean NO2': ':.1f' if 'Exposure Mean NO2' in map_df.columns else False,
        'Exposure Mean OZONE': ':.1f' if 'Exposure Mean OZONE' in map_df.columns else False,
        'Exposure Mean HAP': ':.1f' if 'Exposure Mean HAP' in map_df.columns else False,
    },
    facet_col='Year',
    facet_col_wrap=2,
    facet_col_spacing=0.03,
    facet_row_spacing=0.03,
    color_continuous_scale=[
        [0.0, CUSTOM_PALETTE[14]],
        [0.2, CUSTOM_PALETTE[8]],
        [0.4, CUSTOM_PALETTE[5]],
        [0.6, CUSTOM_PALETTE[3]],
        [0.8, CUSTOM_PALETTE[1]],
        [1.0, CUSTOM_PALETTE[7]],
    ],
    range_color=[0.0, 100.0],
    category_orders={'Year': desired_years},
    title='Population-Weighted Pollution Impact — Global Snapshots (1990, 2000, 2010, 2020)<br><sub>Scaled Total Pollution Burden (0–100)</sub>'
)

fig.for_each_annotation(lambda a: a.update(text=a.text.split('=')[-1]))
fig.update_coloraxes(colorbar_title='Scaled Total<br>Pollution Burden<br>(0–100)')
rows = int(ceil(len(desired_years) / 2))
fig.update_layout(width=1000, height=420 * rows, title_x=0.5, font=dict(size=12))
fig.show()

/var/folders/82/w51l4pcn5351zd0bz7gqwcrr0000gq/T/ipykernel_37553/1928680349.py:16: DeprecationWarning: The library used by the *country names* `locationmode` option is changing in an upcoming version. Country names in existing plots may not work in the new version. To ensure consistent behavior, consider setting `locationmode` to *ISO-3*.
  fig = px.choropleth(


## 4.6 Correlation Matrix

### Definition
The **weighted Pearson correlation** measures the linear association between two variables when each observation carries a different importance (weight).  
In this work, each country–year is weighted by its population, so that the correlation reflects the relationship **as experienced by the average person** rather than the average country.

---

### Formula (step-by-step)

First define the **weighted mean** of a variable \(x\):

$$
\mu_x = \frac{\sum_i w_i x_i}{\sum_i w_i},
\qquad
\mu_y = \frac{\sum_i w_i y_i}{\sum_i w_i}.
$$

The **weighted variance** and **weighted covariance** are:

$$
\operatorname{var}_w(x) = \frac{\sum_i w_i (x_i - \mu_x)^2}{\sum_i w_i},
$$

$$
\operatorname{cov}_w(x,y) = \frac{\sum_i w_i (x_i - \mu_x)(y_i - \mu_y)}{\sum_i w_i}.
$$

thus, the **weighted Pearson correlation** is defined as:

$$
\rho_w(x,y) \;=\;
\frac{\operatorname{cov}_w(x,y)}
     {\sqrt{\operatorname{var}_w(x)\,\operatorname{var}_w(y)}}.
$$

### Fully substituted formula

Expanding everything into a single expression gives:

$$
\rho_w(x,y) \;=\; 
\frac{\sum_i w_i (x_i - \mu_x)(y_i - \mu_y)}
     {\sqrt{\;\sum_i w_i (x_i - \mu_x)^2 \;\;\sum_i w_i (y_i - \mu_y)^2}}
$$


This reduces to the ordinary Pearson correlation if all weights $(w_i)$ are equal.

---

### Why this feature is engineered
Using weighted correlations is essential because the unit of data is a **country–year**, but the scientific question concerns **individual people**.  

- If unweighted, each country contributes equally, regardless of population size.  
- With population weights, larger countries (more people) contribute proportionally more.  

Thus, the weighted correlation answers:  

*“If we picked a random person on Earth, how correlated would their exposure to pollutant A be with their exposure to pollutant B?”*


In [26]:
correlation_colorscale = [
    [0.0,  CUSTOM_PALETTE[8]],
    [0.5,  CUSTOM_PALETTE[5]],
    [0.75, CUSTOM_PALETTE[1]],
    [1.0,  CUSTOM_PALETTE[7]],
]

def heatmap_from_matrix(cm: pd.DataFrame, xlabels: list[str], title: str, show_scale: bool = True):
    z = cm.values
    text = [[f"{v:.2f}" if pd.notnull(v) else '' for v in row] for row in z]
    heat = go.Heatmap(
        z=z, x=xlabels, y=xlabels,
        zmin=-1, zmax=1, colorscale=correlation_colorscale, reversescale=False,
        showscale=show_scale, text=text, texttemplate='%{text}',
        hovertemplate='%{y} vs %{x}: %{z:.2f}<extra></extra>'
    )
    fig = go.Figure(data=[heat])
    fig.update_layout(title=title, width=650, height=520)
    return fig

In [28]:
weight_by_population = True  # False for unweighted


exp_cols = ['Exposure Mean PM25', 'Exposure Mean NO2', 'Exposure Mean OZONE', 'Exposure Mean HAP']
cols_present = [c for c in exp_cols if c in pollutants_data.columns]
short_labels = [col_map[c] for c in cols_present]  # e.g., ["PM2.5","NO2","O3","HAP"]

df = (
    pollutants_data[['Country', 'Year', 'Region Name', *cols_present]]
    .merge(population_data[['Country', 'Year', 'Population']], on=['Country', 'Year'], how='left', validate='m:1')
)

df = df.dropna(subset=cols_present + ['Population', 'Region Name'])

# Weighted Pearson correlation
def calculate_weighted_correlation(data_x: np.ndarray, data_y: np.ndarray, weights: np.ndarray) -> float:
    """
    Calculate the weighted Pearson correlation coefficient between two arrays.
    
    Parameters:
    -----------
    data_x : np.ndarray
        First array of values
    data_y : np.ndarray
        Second array of values
    weights : np.ndarray
        Array of weights corresponding to each pair of observations
        
    Returns:
    --------
    float
        Weighted Pearson correlation coefficient between -1 and 1
        Returns NaN if insufficient valid data or zero variance
    """
    # Convert inputs to float arrays
    weights = np.asarray(weights, dtype=float)
    data_x = np.asarray(data_x, dtype=float)
    data_y = np.asarray(data_y, dtype=float)
    
    # Filter out missing values
    valid_mask = np.isfinite(data_x) & np.isfinite(data_y) & np.isfinite(weights)
    if valid_mask.sum() < 3:  # Need at least 3 points for meaningful correlation
        return np.nan
        
    weights = weights[valid_mask]
    data_x = data_x[valid_mask]
    data_y = data_y[valid_mask]
    
    # Check if weights are valid
    if (weights <= 0).all():
        return np.nan
        
    total_weight = weights.sum()
    if total_weight == 0:
        return np.nan
        
    # Calculate weighted means
    weighted_mean_x = (weights * data_x).sum() / total_weight
    weighted_mean_y = (weights * data_y).sum() / total_weight
    
    # Calculate weighted covariance and variances
    weighted_covariance = (weights * (data_x - weighted_mean_x) * (data_y - weighted_mean_y)).sum() / total_weight
    weighted_variance_x = (weights * (data_x - weighted_mean_x) ** 2).sum() / total_weight
    weighted_variance_y = (weights * (data_y - weighted_mean_y) ** 2).sum() / total_weight
    
    # Check for zero variance
    if weighted_variance_x <= 0 or weighted_variance_y <= 0:
        return np.nan
        
    # Calculate correlation coefficient
    correlation = weighted_covariance / np.sqrt(weighted_variance_x * weighted_variance_y)
    return correlation


def corr_matrix(df, cols, weights = None):
    n = len(cols)
    mat = np.full((n, n), np.nan, dtype=float)
    for i in range(n):
        for j in range(i, n):
            if weights is None:
                r = pd.Series(df[cols[i]]).corr(pd.Series(df[cols[j]]))
            else:
                r = calculate_weighted_correlation(df[cols[i]].values, df[cols[j]].values, weights)
            mat[i, j] = mat[j, i] = r
    return pd.DataFrame(mat, index=cols, columns=cols)

weights_global = df['Population'].values if weight_by_population else None
global_corr = corr_matrix(df, cols_present, weights=weights_global) 



In [29]:
fig = heatmap_from_matrix(
    global_corr, short_labels,
    f"Cross-Pollutant Correlation (Country–Year) — {'Weighted by Population' if weight_by_population else 'Unweighted'}"
)
fig.show()

In [30]:
df['CanonRegion'] = df['Region Name'].map(REGION_MAP).fillna('Other')
desired_order = ['Africa', 'Americas', 'Asia', 'Europe', 'MENA', 'Oceania', 'Other']
regions = [r for r in desired_order if r in set(df['CanonRegion'].unique())]

if regions:
    fig_rg = make_subplots(rows=2, cols=3, subplot_titles=regions)
    r = c = 1
    for region in regions:
        sub = df.loc[df['CanonRegion'] == region, ['Population', *cols_present]]
        w = sub['Population'].values if weight_by_population else None
        cm = corr_matrix(sub, cols_present, weights=w)

        fig_sub = heatmap_from_matrix(
            cm, short_labels,
            title="",
            show_scale=False
        )
        for tr in fig_sub.data:
            fig_rg.add_trace(tr, row=r, col=c)
        fig_rg.update_xaxes(tickangle=45, row=r, col=c)

        c += 1
        if c > 3:
            c = 1
            r += 1


    fig_rg.add_trace(
        go.Heatmap(z=[[0, 1], [1, 0]], showscale=True, zmin=-1, zmax=1,
                   colorscale=correlation_colorscale, reversescale=False, visible=False),
        row=2, col=3
    )
    fig_rg.update_layout(
        height=300 * 2, width=1000,
        title=f"Per-Canonical-Region Cross-Pollutant Correlations — {'Weighted by Population' if weight_by_population else 'Unweighted'}"
    )
    fig_rg.show()


# 5. Mortality Analysis

## 5.1 Mortality Analysis — Global Deaths (1990–2020)

Aggregated total deaths per year across all countries. This provides a global all-cause mortality trend without per-capita normalization.


In [32]:
global_deaths = (
    death_data
    .groupby('Year', as_index=False)['Deaths']
    .sum()
    .query('1990 <= Year <= 2020')
    .sort_values('Year')
)


fig_gd = px.line(
    global_deaths,
    x='Year',
    y='Deaths',
    title='Global All-Cause Deaths (1990–2020)',
)
fig_gd.update_traces(mode='lines+markers', line=dict(width=3), marker=dict(size=6))
fig_gd.update_layout(
    xaxis_title='Year',
    yaxis_title='Total Deaths',
    template='plotly_white',
    hovermode='x unified',
)
fig_gd.update_xaxes(dtick=2)

fig_gd

## 5.2 Year-over-Year (YoY) Increase in Global Deaths

This view shows two metrics derived from globally aggregated deaths by year:

- Absolute change vs. previous year (left axis)
- Percentage change vs. previous year (right axis)

Both are computed directly from the existing `global_deaths` time series and visualized with the notebook’s established color palette.

In [33]:
_yoy = (
    global_deaths[['Year', 'Deaths']]
    .sort_values('Year')
)

_yoy['Abs Change'] = _yoy['Deaths'].diff()

_yoy['Pct Change'] = _yoy['Deaths'].pct_change() * 100.0

yoy_deaths = _yoy

In [ ]:
fig_yoy = make_subplots(specs=[[{"secondary_y": True}]])

fig_yoy.add_trace(
    go.Scatter(
        x=yoy_deaths['Year'],
        y=yoy_deaths['Abs Change'],
        name='Absolute Change',
        mode='lines+markers',
        line=dict(color=CUSTOM_PALETTE[10], width=2),
        marker=dict(size=6)
    ),
    secondary_y=False,
)

fig_yoy.add_trace(
    go.Scatter(
        x=yoy_deaths['Year'],
        y=yoy_deaths['Pct Change'],
        name='Percentage Change',
        mode='lines+markers',
        line=dict(color=CUSTOM_PALETTE[8], width=2),
        marker=dict(size=6)
    ),
    secondary_y=True,
)

fig_yoy.update_layout(
    title='Year-over-Year Change in Global Deaths',
    legend_title_text='',
    hovermode='x unified',
    template="plotly_white",
)

fig_yoy.update_xaxes(title_text='Year')
fig_yoy.update_yaxes(title_text='Increase in Total Deaths', secondary_y=False)
fig_yoy.update_yaxes(title_text='Year-on-Year % Change', secondary_y=True)

fig_yoy.show()

Notes

- Absolute change is the difference in total deaths from year t to t-1.
- Percentage change is the relative change vs t-1 expressed in percent.
- The first year shows NaN because no prior year exists by design.

## 5.3 Is the world growing or dying overall?
Global CDR, PGR, and Gap (1990–2020)

For that we would need the following features: 

### 1. Crude Death Rate (CDR)

**Definition:** Number of deaths per 1,000 population in a given country-year.  

**Formula:**

$$
\text{CDR} = 1000 \times \frac{\text{Deaths}}{\text{Population}}
$$

**Rationale:** Normalizes deaths relative to population size, enabling fair comparison across countries of different sizes.

---

### 2. Population Growth Rate (PGR)

**Definition:** Year-over-year percentage growth in population scaled to deaths-per-1,000 units for comparability with CDR

**Formula:**

$$
PGR_{\%} = 100 \times \frac{\text{Population}_{t} - \text{Population}_{t-1}}{\text{Population}_{t-1}}
$$

$$
\text{PGR}_{\text{per 1000}} = 10 \times PGR_{\%}
$$

**Rationale:** Measures how fast a population is growing or shrinking annually, while aligning units with CDR, allowing a direct “growth vs mortality” gap analysis.

---

### 4. Growth–Mortality Gap (Gap)

**Definition:** Net balance of population growth versus crude death rate (per 1,000).  

**Formula:**

$$
\text{Gap} = \text{PGR}_{\text{per 1000}} - \text{CDR}
$$

**Rationale:** Indicates whether a country’s population is growing faster than deaths (positive) or being outweighed by mortality (negative).


We’ll compute the three series, estimate Theil–Sen trends, and detect turning points (change-points) on CDR.

In [34]:
global_population = (
    population_data
    .groupby('Year', as_index=False)['Population']
    .sum()
    .query('1990 <= Year <= 2020')
    .sort_values('Year')
)


combined_global = (
    global_deaths[['Year', 'Deaths']]
    .merge(global_population[['Year', 'Population']], on='Year', how='inner', validate='1:1')
    .sort_values('Year')
)


combined_global['global_cdr'] = 1000.0 * combined_global['Deaths'] / combined_global['Population']
combined_global['global_pgr'] = 10*(100.0 * (combined_global['Population'].diff()) / combined_global['Population'].shift(1))
combined_global['global_gap'] = combined_global['global_pgr'] - combined_global['global_cdr']

years = combined_global['Year'].values


x = combined_global['Year'].values

def ts_trend(y):
    mask = np.isfinite(y)
    slope, intercept, lo, hi = theilslopes(y[mask], x[mask], 0.95)
    return slope, intercept, lo, hi

cdr_slope, cdr_intercept, cdr_lo, cdr_hi = ts_trend(combined_global['global_cdr'].values)
pgr_slope, pgr_intercept, pgr_lo, pgr_hi = ts_trend(combined_global['global_pgr'].values)
gap_slope, gap_intercept, gap_lo, gap_hi = ts_trend(combined_global['global_gap'].values)


yhat_cdr = cdr_intercept + cdr_slope * x
yhat_pgr = pgr_intercept + pgr_slope * x
yhat_gap = gap_intercept + gap_slope * x



In [37]:
fig = go.Figure()


fig.add_trace(go.Scatter(
    x=years, y=combined_global['global_cdr'],
    mode='lines+markers',
    name='Crude Death Rate',
    line=dict(color=CUSTOM_PALETTE[7], width=3),
    marker=dict(size=6)
))
fig.add_trace(go.Scatter(
    x=years, y=combined_global['global_pgr'],
    mode='lines+markers',
    name='Population Growth Rate',
    line=dict(color=CUSTOM_PALETTE[8], width=3),
    marker=dict(size=6)
))



cdr_pos = np.where(combined_global['global_pgr'] >= combined_global['global_cdr'], combined_global['global_cdr'], np.nan)
pgr_pos = np.where(combined_global['global_pgr'] >= combined_global['global_cdr'], combined_global['global_pgr'], np.nan)
cdr_neg = np.where(combined_global['global_cdr'] > combined_global['global_pgr'], combined_global['global_cdr'], np.nan)
pgr_neg = np.where(combined_global['global_cdr'] > combined_global['global_pgr'], combined_global['global_pgr'], np.nan)


fig.add_trace(go.Scatter(
    x=years, y=cdr_pos, mode='lines',
    line=dict(width=0), hoverinfo='skip', showlegend=False
))
fig.add_trace(go.Scatter(
    x=years, y=pgr_pos, mode='lines',
    line=dict(width=0), hoverinfo='skip',
    fill='tonexty', fillcolor='rgba(46, 204, 113, 0.18)',  # soft green
    name='Gap: Growth > Death'
))


fig.add_trace(go.Scatter(
    x=years, y=pgr_neg, mode='lines',
    line=dict(width=0), hoverinfo='skip', showlegend=False
))
fig.add_trace(go.Scatter(
    x=years, y=cdr_neg, mode='lines',
    line=dict(width=0), hoverinfo='skip',
    fill='tonexty', fillcolor='rgba(231, 76, 60, 0.18)',   # soft red
    name='Gap: Death > Growth'
))

fig.add_trace(go.Scatter(
    x=years, y=yhat_cdr, mode='lines',
    name=f'CDR trend (Theil–Sen: {cdr_slope:.3f}/yr)',
    line=dict(color=CUSTOM_PALETTE[7], dash='dash'),
))
fig.add_trace(go.Scatter(
    x=years, y=yhat_pgr, mode='lines',
    name=f'PGR trend (Theil–Sen: {pgr_slope:.3f}/yr)',
    line=dict(color=CUSTOM_PALETTE[8], dash='dash'),
))
fig.add_trace(go.Scatter(
     x=[None], y=[None], mode='lines',
    name=f'Gap trend (Theil–Sen: {gap_slope:.3f}/yr)',
    line=dict(color="#FFFFFF", dash='dot'),
))

fig.update_yaxes(title_text='per 1,000')
fig.update_xaxes(title_text='Year')

fig.update_layout(
    title='Global CDR vs PGR with Shaded Gap & Trends (1990–2020)',
    hovermode='x unified',
    template='plotly_white',
    legend_title_text='Series',
    
)

fig

Unfortunately we see, that the Growth rate is decreasing at an alarming 18% every year over the last three decades, if the current rate of decrease in the growth rate keeps it's trend, by 2040, we will end up with only 0.19% growth rate. 

## 5.3.1 Top-10 Countries by Growth–Mortality Gap

This section ranks countries by the Growth–Mortality Gap at the most recent year available in this notebook’s workflow.

- CDR (Crude Death Rate): total deaths per 1,000 population
- PGR per 1,000: year-over-year population growth scaled to per-1,000 units
- Gap = PGR per 1,000 − CDR

Interpretation:
- Highest gap: population growth outpaces mortality burden (faster growing relative to mortality)
- Lowest gap: mortality burden dominates relative to growth (higher mortality burden relative to growth)


In [38]:
deaths_agg = death_data.groupby(['Country', 'Year'], as_index=False)['Deaths'].sum()

country_year = deaths_agg.merge(population_data, on=['Country', 'Year'], how='inner')


country_year['CDR'] = (country_year['Deaths'] / country_year['Population']) * 1000.0


country_year = country_year.sort_values(['Country', 'Year'])
country_year['PGR_per_1000'] = 10 * (
    100.0 * country_year.groupby('Country')['Population'].diff() / country_year.groupby('Country')['Population'].shift(1)
)

country_year['Gap'] = country_year['PGR_per_1000'] - country_year['CDR']


In [39]:
gap_5y = (
    country_year.loc[country_year['Year'].between(2016, 2020), ['Country', 'Gap']]
    .groupby('Country', as_index=False)
    .mean()
    .rename(columns={'Gap': 'Gap_5Y_Avg'})
)

gap_10y = (
    country_year.loc[country_year['Year'].between(2000, 2020), ['Country', 'Gap']]
    .groupby('Country', as_index=False)
    .mean()
    .rename(columns={'Gap': 'Gap_10Y_Avg'})
)

gap_30y = (
    country_year.loc[country_year['Year'].between(1991, 2020), ['Country', 'Gap']]
    .groupby('Country', as_index=False)
    .mean()
    .rename(columns={'Gap': 'Gap_30Y_Avg'})
)

Choropleth here

In [40]:

mean_5y = np.mean(gap_5y['Gap_5Y_Avg'])
mean_10y = np.mean(gap_10y['Gap_10Y_Avg'])
mean_30y = np.mean(gap_30y['Gap_30Y_Avg'])


fig = make_subplots(rows=3, cols=1, 
                    subplot_titles=('5-Year Average Gap (2016-2020)',
                                   '20-Year Average Gap (2000-2020)', 
                                   '30-Year Average Gap (1991-2020)'),
                    vertical_spacing=0.1,
                    shared_xaxes=True)

# 5-year average gap (2016-2020)
fig.add_trace(
    go.Histogram(
        x=gap_5y['Gap_5Y_Avg'],
        nbinsx=30,
        marker=dict(
            color=CUSTOM_PALETTE[0],
            line=dict(
                color="gray",
                width=1  
            )
        ),
        opacity=0.8,
        name='5-Year Avg'
    ),
    row=1, col=1
)

# 20-year average gap (2000-2020)
fig.add_trace(
    go.Histogram(
        x=gap_10y['Gap_10Y_Avg'],
        nbinsx=30,
       marker=dict(
            color=CUSTOM_PALETTE[2],
            line=dict(
                color="gray",   # outline color
                width=1         # outline thickness
            )
        ),
        opacity=0.8,
        name='20-Year Avg'
    ),
    row=2, col=1
)

# 30-year average gap (1991-2020)
fig.add_trace(
    go.Histogram(
        x=gap_30y['Gap_30Y_Avg'],
        nbinsx=30,
        marker=dict(
            color=CUSTOM_PALETTE[4],
            line=dict(
                color="gray",
                width=1
            )
        ),
        opacity=0.8,
        name='30-Year Avg'
    ),
    row=3, col=1
)

fig.add_vline(
    x=0,
    line_width=2,
    line_dash="dash",
    line_color="red",
    row=1, col=1
)
fig.add_vline(
    x=0,
    line_width=2,
    line_dash="dash",
    line_color="red",
    row=2, col=1
)
fig.add_vline(
    x=0,
    line_width=2,
    line_dash="dash",
    line_color="red",
    row=3, col=1
)


fig.add_vline(
    x=mean_5y,
    line_width=2,
    line_dash="solid",
    line_color="blue",
    annotation_text=f"Mean = {mean_5y:.2f}",
    annotation_position="top left",
    row=1, col=1
)

fig.add_vline(
    x=mean_10y,
    line_width=2,
    line_dash="solid",
    line_color="blue",
    annotation_text=f"Mean = {mean_10y:.2f}",
    annotation_position="top left",
    row=2, col=1
)

fig.add_vline(
    x=mean_30y,
    line_width=2,
    line_dash="solid",
    line_color="blue",
    annotation_text=f"Mean = {mean_30y:.2f}",
    annotation_position="top left",
    row=3, col=1
)


fig.update_layout(
    height=800,
    title_text='Distribution of Growth-Mortality Gap across Time Periods',
    showlegend=False,
    template='plotly_white'
)

fig.update_xaxes(title_text='Growth-Mortality Gap (per 1,000)', row=3, col=1)
fig.update_yaxes(title_text='Number of Countries')

fig.show()

In [41]:
gap_data = (
    gap_5y.merge(gap_10y, on='Country', how='outer')
          .merge(gap_30y, on='Country', how='outer')
)

gap_long = gap_data.melt(
    id_vars=['Country'],
    value_vars=['Gap_5Y_Avg', 'Gap_10Y_Avg', 'Gap_30Y_Avg'],
    var_name='Period',
    value_name='Gap'
)


gap_long['Period'] = gap_long['Period'].replace({
    'Gap_5Y_Avg': '5-Year Average (2016-2020)',
    'Gap_10Y_Avg': '20-Year Average (2000-2020)',
    'Gap_30Y_Avg': '30-Year Average (1991-2020)'
})


gap_colorscale = [
        [0.0, CUSTOM_PALETTE[7]],
        [0.25, CUSTOM_PALETTE[1]],
        [0.5, "white"],          
        [0.75, CUSTOM_PALETTE[5]],
        [1.0, CUSTOM_PALETTE[8]],
    ]

# Create choropleth map
fig = px.choropleth(
    gap_long,
    locations='Country',
    locationmode='country names',
    color='Gap',
    facet_col='Period',
    facet_col_wrap=1,
    color_continuous_scale=gap_colorscale,
    range_color=[-35, 35],
    labels={'Gap': 'Growth-Mortality Gap'},
    title='Growth-Mortality Gap by Country (PGR - CDR per 1,000 population)',
    height=1200,
    width=1000
)

# Clean up facet titles
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))

# Add zero reference line to color scale
fig.update_layout(
    coloraxis_colorbar=dict(
        title='Growth-Mortality Gap',
        tickvals=[0],
        ticktext=['0 (Growth = Death)']
    )
)

fig.show()

/var/folders/82/w51l4pcn5351zd0bz7gqwcrr0000gq/T/ipykernel_37553/3510174836.py:30: DeprecationWarning: The library used by the *country names* `locationmode` option is changing in an upcoming version. Country names in existing plots may not work in the new version. To ensure consistent behavior, consider setting `locationmode` to *ISO-3*.
  fig = px.choropleth(


In [ ]:
top_10 = gap_30y.nlargest(10, "Gap_30Y_Avg")
bottom_10 = gap_30y.nsmallest(10, "Gap_30Y_Avg")

In [ ]:
top_10

,Country,Gap_30Y_Avg
144,Qatar,62.058914
184,United Arab Emirates,52.584008
56,Equatorial Guinea,35.103031
90,Jordan,34.112716
13,Bahrain,32.152980
153,Saudi Arabia,31.407078
94,Kuwait,31.322481
0,Afghanistan,30.386627
132,Oman,28.735686
194,Yemen,25.107721


In [ ]:
bottom_10

,Country,Gap_30Y_Avg
97,Latvia,-28.312245
27,Bulgaria,-27.011994
102,Lithuania,-24.814192
155,Serbia,-24.048345
66,Georgia,-23.258953
146,Republic of Moldova,-22.346751
183,Ukraine,-22.045408
42,Croatia,-21.784269
23,Bosnia and Herzegovina,-21.136478
147,Romania,-21.057533


## Cause of Death: (Top 10 Causes)

In [46]:
cause_death_totals = (
    death_data
    .groupby('Cause')['Deaths']
    .sum()
    .reset_index()
)

cause_death_totals['mortality_share'] = cause_death_totals['Deaths'] / cause_death_totals['Deaths'].sum()

fig = px.treemap(
    cause_death_totals,
    path=['Cause'],
    values='mortality_share',
    color='mortality_share',
    color_continuous_scale=[
        [0.0, CUSTOM_PALETTE[14]],
        [0.2, CUSTOM_PALETTE[8]],
        [0.4, CUSTOM_PALETTE[5]],
        [0.6, CUSTOM_PALETTE[3]],
        [0.8, CUSTOM_PALETTE[1]],
        [1.0, CUSTOM_PALETTE[7]],
    ],
    title="Causes of Death Sized by Mortality Share (Excluding 'All Causes')"
)

fig.update_traces(
    hovertemplate='<b>Cause:</b> %{label}<br>Share: %{value:.2%}<br>Total Deaths: %{customdata[0]:,}<extra></extra>',
    customdata=cause_death_totals[['Deaths']]
)
fig.update_layout(margin=dict(t=60, l=0, r=0, b=0))

fig.show()

In [48]:
top_10_diseases = (
    death_data
    .groupby('Cause')['Deaths']
    .sum()
    .sort_values(ascending=False)
    .head(10)
)

fig = px.bar(
    x=top_10_diseases.index,
    y=top_10_diseases.values,
    labels={"x": "Disease", "y": "Total Deaths"},
    title="Top 10 Diseases by Total Death Count (Excluding 'All Causes')",
    text=top_10_diseases.values
)

fig.update_layout(
    xaxis_title="Disease",
    yaxis_title="Total Deaths",
    hovermode="x unified",
    height=600,
    template="plotly_white",
    xaxis=dict(tickangle=45),
)


fig.update_traces(
    texttemplate='%{y:,.0f}',
    textposition='outside',
    hovertemplate='<b>%{x}</b><br>Total Deaths: %{y:,.0f}<extra></extra>'
)

fig.show()

# Conclusion: Air Pollution, Mortality, and Global Health Patterns (1990-2020)

This comprehensive exploratory analysis reveals a world in environmental and demographic transition, where progress in reducing traditional pollutants exists alongside emerging challenges and stark global inequalities. From 1990 to 2020, we observe a remarkable success story in pollution control—household air pollution decreased by 36%, PM2.5 by 21%, and NO₂ by 18%—demonstrating that coordinated global efforts can yield significant environmental improvements. However, this progress is overshadowed by the alarming 13.5% increase in ozone levels, signaling a new frontier in air quality management that requires urgent attention.

The geographical distribution of pollution burden reveals a troubling pattern of environmental injustice. While Sub-Saharan African nations like Niger, Mali, and Burkina Faso face the highest pollution intensity per capita, it is the densely populated developing countries—India, China, Pakistan, Nigeria, and Bangladesh—that bear the greatest absolute human impact when population size is considered. This analysis exposes a critical insight: the countries with the most people exposed to dangerous pollution levels are precisely those undergoing rapid economic development, creating a perfect storm of environmental and public health challenges.

Perhaps most concerning is the demographic backdrop against which these pollution patterns unfold. Global population growth rates are declining at an unprecedented 18% annually, projecting a world growth rate of merely 0.19% by 2040 if current trends continue. This demographic shift, combined with the strong correlations observed between different pollutants (particularly PM2.5, NO₂, and HAP), suggests that countries struggling with air quality face compound challenges that require comprehensive, multi-pollutant interventions rather than piecemeal approaches.

The mortality analysis reveals profound regional disparities in the balance between population growth and death rates. While Gulf states demonstrate robust growth-mortality gaps driven by economic prosperity, Eastern European and post-Soviet nations show concerning patterns where mortality increasingly outpaces population growth. These demographic fault lines, intersecting with pollution exposure patterns, create a complex global health landscape where environmental interventions must be carefully calibrated to local demographic and economic realities.

**Ultimately, this analysis demonstrates that air pollution and mortality are not merely environmental or health issues—they are fundamental challenges of global equity and sustainable development. The data compels us toward a future where pollution reduction efforts must prioritize population-weighted impact, recognizing that effective global health policy lies not just in cleaning the most polluted places, but in protecting the greatest number of people from preventable environmental harm.**